# E18 CPU harness (Kaggle pinned image)

The fake-server harness of the E18 notebook (S1-60K on vLLM 0.32, 20 streams), all 6 scenarios on the same image as the GPU notebook. Kit: `foysalemonshanto/q27-e18-cpu-kit` (q27x_e18_cpu_kit_ab9239001517.bin).


In [ ]:
# E18 CPU harness on Kaggle (pinned image = the GPU notebook's): the fake-server harness, all 6 scenarios
import hashlib, io, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
KIT_NAME, KIT_SHA = "q27x_e18_cpu_kit_ab9239001517.bin", "ab9239001517e0d65949a2e3cbe14062b53c7b26b005c2948299c8a0000c9558"
NB = "Q27-XHIGH-E18_S1_60K_VLLM032_S20_R2.ipynb"
MODES = ["mixed", "kvgate", "hardstop", "rerun", "lmofail", "allfail", "death"]
WH = "/kaggle/input/datasets/mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3"
OUT = Path("/kaggle/working"); K = Path("/tmp/e18_kit"); T0 = time.time()
REPORT = {"kit": KIT_NAME, "python": sys.version, "nb_sha": None, "harness": {}}


def save():
    (OUT / "cpu_harness_report.json").write_text(json.dumps(REPORT, indent=1), encoding="utf-8")


def find(name, sha):
    for pat in (name, "*/" + name, "*/*/" + name, "*/*/*/" + name, "*/*/*/*/" + name):
        for f in sorted(Path("/kaggle/input").glob(pat)):
            b = f.read_bytes()
            if hashlib.sha256(b).hexdigest() == sha:
                return b
    raise FileNotFoundError(f"{name} ({sha[:12]}) not under /kaggle/input: attach foysalemonshanto/q27-e18-cpu-kit")


shutil.rmtree(K, ignore_errors=True)
with zipfile.ZipFile(io.BytesIO(find(KIT_NAME, KIT_SHA))) as z:
    z.extractall(K)
man = json.loads((K / "MANIFEST.json").read_text())
bad = [k for k, h in man.items() if hashlib.sha256((K / k).read_bytes()).hexdigest() != h]
assert not bad, bad
REPORT["nb_sha"] = hashlib.sha256((K / "nb" / NB).read_bytes()).hexdigest()
print("kit OK:", len(man), "files |", sys.version.split()[0], "| notebook", REPORT["nb_sha"], flush=True)
# the GPU notebook's tokenizer pins (its CELL 2), from the S1 wheelhouse
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", WH,
                    "transformers==4.57.6", "tokenizers==0.22.2", "huggingface-hub==0.36.2"], capture_output=True, text=True)
REPORT["pins"] = {"rc": r.returncode, "tail": (r.stdout + r.stderr)[-800:]}
print("pins:", r.returncode, flush=True)
ENV = dict(os.environ, E18_DATA=str(K / "data"), E18_TOKDIR=str(K / "tok"), PYTHONIOENCODING="utf-8")
for mode in MODES:
    t = time.time()
    try:
        r = subprocess.run([sys.executable, str(K / "harness/harness_e18_nb.py"), str(K / "nb" / NB), mode],
                           cwd=K / "harness", env=ENV, capture_output=True, text=True, timeout=1800)
        out, rc = r.stdout + r.stderr, r.returncode
    except subprocess.TimeoutExpired as e:
        out, rc = (e.stdout or "") if isinstance(e.stdout, str) else "", "timeout"
    (OUT / f"harness_{mode}.log").write_text(out, encoding="utf-8")
    lines = out.splitlines()
    REPORT["harness"][mode] = {"rc": rc, "sec": round(time.time() - t), "ok": rc == 0 and "ALL CHECKS PASS" in out,
                               "passes": sum(1 for x in lines if x.startswith("PASS")),
                               "summary": [x for x in lines if "ALL CHECKS PASS" in x or " failed in " in x],
                               "fails": [x[:1500] for x in lines if x.startswith("FAIL")],
                               "traceback_tail": out[-3000:] if "Traceback" in out else ""}
    save()
    print(mode, "->", REPORT["harness"][mode]["passes"], "pass", REPORT["harness"][mode]["summary"],
          REPORT["harness"][mode]["fails"][:6], flush=True)
REPORT["sec"] = round(time.time() - T0)
REPORT["all_ok"] = all(v["ok"] for v in REPORT["harness"].values())
save()
print("ALL OK" if REPORT["all_ok"] else "NOT OK", "| total", REPORT["sec"], "s", flush=True)
